# P7. 추출 프롬프트 v2 + LLM 추출 시험

## 1. 이 노트북에서 하는 일
- 블록 8개를 LLM에 넣고, **학교가 하는 행동만** 정해진 양식으로 받습니다.
- 블록의 **소제목**(예: "학교[담당교사] 역할")을 함께 줘서 역할을 판단할 근거로 씁니다.
- 근거 문장은 **블록 안에 정확히 있는지** 먼저 확인합니다. 그다음 쪽 단위 후보 확인을 따로 표시합니다.
- 오류가 난 블록은 "0건"이 아니라 **오류**로 따로 기록합니다.
- 결과는 `steps_raw_v2.jsonl`에 저장합니다. 이미 사람이 검수한(approved/rejected) 행은 지우지 않습니다.

## 2. 핵심 생각 (쉽게)
- **LLM은 찾고, 코드는 확인합니다.** 다만 코드 확인도 **후보 확인**일 뿐입니다. 맞는지는 사람이 원문을 보고 판단합니다.
- **학교 행동만 뽑습니다.** 의료기관·보건소가 하는 일은 학교 행동이 아닙니다.
- **역할은 근거가 있을 때만 씁니다.** 원문이나 소제목에 역할 근거가 없으면 역할을 비워 둡니다(`null`).
- **"반드시"는 지침 표현 강도입니다.** 법적 의무로 단정하지 않습니다.

## 3. 블록 불러오기

In [1]:
import json
from preprocess.prevention_legal.pages import DOCS, excerpt_in_page

# 블록 파일(블록 만들기 노트북 02의 결과)을 한 줄씩 읽어 목록으로 만든다
BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"
blocks = [json.loads(line) for line in open(BLOCKS, encoding="utf-8")]

# 문서 ID(예: kdca_measles_2026)로 쪽 읽기용 키(예: measles)를 찾는 표
key_of = {doc_id: key for key, (doc_id, _) in DOCS.items()}

print("블록", len(blocks), "개")
for b in blocks:
    # 보이지 않는 공백 문자(zero-width space)를 지우고 앞 40자만 출력
    print(b["chunk_id"], "|", b["text"][:40].replace("​", ""))


블록 8 개
kdca_measles_2026:p0093:b01 | 1. 교내 **홍역(의심)환자** **[*]** **발생 시 반드시 의료
kdca_measles_2026:p0093:b02 | - 고열, 발진과 함께 기침, 콧물, 결막염 등의 증상이 동반하는 경우
kdca_measles_2026:p0093:b03 | 3. 홍역(의심) 진단을 받지 않았으나 **유사증상자가 2주 이내 2명 
kdca_measles_2026:p0024:b01 | (의사)환자를 진단한 경우
발생
24시간 이내에 관할보건소로 신고
moleg_infectious_disease_act_21521:p0018:b01 | 제11조(의사 등의 신고) ① 의사, 치과의사 또는 한의사는 다음 각 호
moleg_infectious_disease_act_21521:p0018:b02 | ③ 제1항 및 제2항에 따라 보고를 받은 의료기관의 장 및 제16조의2에
moleg_infectious_disease_act_21521:p0019:b01 | 제12조(그 밖의 신고의무자) ① 다음 각 호의 어느 하나에 해당하는 사
moleg_infectious_disease_act_21521:p0019:b02 | 2. 학교, 사회복지시설, 병원, 관공서, 회사, 공연장, 예배장소, 선


## 4. 결과 양식 (추출 후보용 일부 필드)
스키마 v3.1에서 추출 후보로 쓰는 필드만 모았습니다. 완전한 스키마가 아닙니다. 형식(허용값)만 확인하고, 의미는 사람이 검수합니다.

In [2]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

# 스키마 v3.1 허용값 (추출 후보용 일부 필드)
Stage = Literal["PREVENTION", "RESPONSE_1", "RESPONSE_2", "RESPONSE_3", "RECOVERY",
                "ALL_STAGES", "NATIONAL_CRISIS", "UNSPECIFIED"]
Scope = Literal["COMMON", "DISEASE_GROUP", "NAMED_DISEASES", "UNKNOWN"]
Modality = Literal["MANDATORY", "RECOMMENDED", "DISCRETIONARY", "UNSPECIFIED"]
Basis = Literal["EXPLICIT", "EXAMPLE"]

class SchoolActionCand(BaseModel):
    """학교가 하는 행동 한 개 (추출 후보). 형식만 확인하며 원문의 의미는 사람이 검수한다."""
    action_text: str = Field(description="학교가 하는 행동을 한 문장으로")
    stage: Stage = Field(description="원문이 배치한 단계. 불명확하면 UNSPECIFIED")
    scope: Scope = Field(description="COMMON / DISEASE_GROUP / NAMED_DISEASES / UNKNOWN")
    when_text: str = Field(description="적용 조건 (원문 표현)")
    timing_text: Optional[str] = Field(default=None, description="기한이 원문에 있으면 그 표현, 없으면 null")
    modality: Modality = Field(description="원문 표현 강도. 법적 의무로 단정하지 않음")
    role_name: Optional[str] = Field(default=None, description="역할. 문맥 소제목에 근거할 때만, 없으면 null")
    basis: Optional[Basis] = Field(default=None, description="역할 근거: 명시면 EXPLICIT, 예시면 EXAMPLE, 역할이 없으면 null")
    evidence_quote: str = Field(description="블록 원문 행을 그대로 복사한 근거 문장")

class ExtractionResult(BaseModel):
    actions: list[SchoolActionCand] = Field(description="학교 행동이 없으면 빈 목록")

print("스키마 준비 완료")


스키마 준비 완료


## 5. 프롬프트 v2
v1에서 빠졌던 규칙을 넣었습니다. 특히 **소제목을 역할 판단에만 쓰고**, 근거 문장에는 넣지 않습니다.

In [3]:
PROMPT_V2 = """주어진 원문과 제목·소제목 문맥에서 학교가 수행하는 행동만 추출한다.
서로 다른 행동은 각각 분리하고 조건을 보존한다.
조건이 여러 개면(예: '진단을 받지 않았으나 … 발생할 경우') 앞 조건도 빠뜨리지 않고 when_text에 옮긴다.
학교가 의료기관 진료를 안내하거나 보건소에 신고하는 행동은 포함한다.
의료기관·보건소가 직접 수행하는 행동은 제외한다.
역할은 원문 또는 제공된 소제목에 근거할 때만 기록하고, 소제목의 수행 주체를 그대로 옮긴다(예: 학교[담당교사]). 역할의 기본값은 EXAMPLE이다. 원문이나 소제목이 수행 주체를 명시하고 그 아래 행동을 직접 나열할 때만 EXPLICIT을 쓴다. 예를 들어 소제목이 '학교[담당교사] 역할'이고 그 아래에 학교가 할 행동이 직접 적혀 있으면 EXPLICIT이다. 예시·자율 조정 단서가 있으면 EXAMPLE이다.
근거 문장은 블록 원문에서 그대로 복사한다. 소제목은 근거 문장에 넣지 않고 역할 판단에만 쓴다.
stage는 원문이 배치한 단계만 사용하고, 불명확하면 UNSPECIFIED로 둔다.
질병명·강도·기한은 원문에 근거해 기록한다. 원문이 '반드시'라고 써도 법적 의무로 단정하지 않는다.
근거가 없거나 판단이 어려운 역할·기한은 null로 둔다.
행동이 없으면 빈 목록을 반환한다.
"""
print(len(PROMPT_V2), "자 프롬프트 v2 준비")


648 자 프롬프트 v2 준비


## 6. LLM 추출 실행 (블록 8개, 소제목 포함)

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv, find_dotenv
# 저장소 루트의 .env에서 API 키를 읽는다 (노트북 위치에서 위로 올라가며 찾음)
load_dotenv(find_dotenv(usecwd=True), override=True)

# 모델을 만들고, 출력이 ExtractionResult 형식이 되도록 고정한다
model = init_chat_model("openai:gpt-6-luna", reasoning_effort="none")
extractor = model.with_structured_output(ExtractionResult)

results = []   # (블록, 상태, 행동 목록, 오류 이름)
for b in blocks:
    # 소제목은 역할 판단에만 쓰도록 사람 메시지에 따로 넣는다
    section = b.get("section") or "(소제목 없음)"
    human = ("문서: " + b["document_id"] + " / " + str(b["pdf_page"]) + "쪽 / 블록 " + b["chunk_id"]
             + "\n소제목(문맥): " + section + "\n\n원문:\n" + b["text"])
    # 블록 하나가 실패해도 다음 블록은 계속 처리한다
    try:
        out = extractor.invoke([("system", PROMPT_V2), ("human", human)])
        results.append((b, "ok", out.actions, None))
        print(f"{b['chunk_id']} -> 학교 행동 {len(out.actions)}개")
    except Exception as e:
        results.append((b, "error", [], type(e).__name__))
        print(f"{b['chunk_id']} -> 오류 {type(e).__name__} (재실행 대상)")


kdca_measles_2026:p0093:b01 -> 학교 행동 2개


kdca_measles_2026:p0093:b02 -> 학교 행동 0개


kdca_measles_2026:p0093:b03 -> 학교 행동 1개


kdca_measles_2026:p0024:b01 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0018:b01 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0018:b02 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0019:b01 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0019:b02 -> 학교 행동 0개


## 7. 원문 확인 (2단계)
- **블록 일치:** 근거 문장이 해당 블록 안에 정확히 있음 → 가장 믿을 만함
- **쪽 후보:** 블록에는 없고 같은 쪽 어딘가에 비슷한 글이 있음 → 원문을 직접 보고 판단
- **확인 필요:** 두 검사에서 모두 찾지 못함 → 원문을 직접 보고 다시 추출하거나 버림

두 번째 검사(쪽 후보)는 공백과 일부 기호를 지우고 비교하므로, 줄바꿈이나 표 셀을 이어 붙인 문장도 걸릴 수 있습니다. 그래서 `쪽 후보`는 확정이 아니라 후보입니다.

In [5]:
# 원문 확인 (블록별 오류를 따로 기록)
#  1) 블록 안 정확 일치 (가장 엄격)  -> "블록 일치"
#  2) 쪽 단위 후보 확인 (느슨함)     -> "쪽 후보" (사람이 원문을 직접 봐야 함)
#  3) 두 검사 모두 찾지 못함          -> "확인 필요"
import re, uuid
RUN_ID = uuid.uuid4().hex[:8]   # 이번 실행을 구분하는 ID
rows = []
for b, status, actions, err in results:
    if status == "error":
        rows.append({"run_id": RUN_ID, "chunk_id": b["chunk_id"], "status": "error",
                     "error": err, "review_status": "unreviewed"})
        continue
    try:
        key = key_of[b["document_id"]]
    except KeyError as e:
        rows.append({"run_id": RUN_ID, "chunk_id": b["chunk_id"], "status": "error",
                     "error": f"등록되지 않은 문서 ID: {e}", "review_status": "unreviewed"})
        continue
    for i, a in enumerate(actions, 1):
        try:
            q = a.evidence_quote.strip()
            # 글자·숫자가 없는 인용문(예: "**")은 쪽 검사에서도 무조건 맞아 버리므로 먼저 걸러낸다
            if not re.search(r"\w", q):
                check = "확인 필요"
            elif q in b["text"]:
                check = "블록 일치"
            elif excerpt_in_page(q, key, b["pdf_page"]):
                check = "쪽 후보"
            else:
                check = "확인 필요"
        except Exception as e:
            rows.append({"run_id": RUN_ID, "chunk_id": b["chunk_id"], "status": "error",
                         "error": type(e).__name__, "review_status": "unreviewed"})
            continue
        rows.append({
            "run_id": RUN_ID,
            "chunk_id": b["chunk_id"],
            "action_no": i,
            "status": "ok",
            "action_text": a.action_text,
            "stage": a.stage, "scope": a.scope, "modality": a.modality,
            "when_text": a.when_text, "timing_text": a.timing_text,
            "role_name": a.role_name, "basis": a.basis,
            "evidence_quote": a.evidence_quote,
            "quote_check": check,
            "context_section": b.get("section", ""),
            "review_status": "unreviewed",
        })

ok_rows = [r for r in rows if r["status"] == "ok"]
print("실행 ID:", RUN_ID, "| 학교 행동 후보:", len(ok_rows), "개 | 오류 블록:", sum(1 for r in rows if r["status"] == "error"))
for r in ok_rows:
    print(f"[{r['quote_check']}] {r['chunk_id']} | {r['action_text'][:40]} | 역할 {r['role_name']} ({r['basis']})")


실행 ID: ed4ab2cc | 학교 행동 후보: 3 개 | 오류 블록: 0
[블록 일치] kdca_measles_2026:p0093:b01 | 홍역(의심)환자가 발생하면 의료기관에서 진료를 받도록 안내한다. | 역할 학교[담당교사] (EXPLICIT)
[블록 일치] kdca_measles_2026:p0093:b01 | 관할 보건소에 신고한다. | 역할 학교[담당교사] (EXPLICIT)
[블록 일치] kdca_measles_2026:p0093:b03 | 보건소로 신고한다. | 역할 학교[담당교사] (EXPLICIT)


## 8. 저장 (검수 전 상태, 검수된 행은 보존)
- 사람이 검수한(approved/rejected) 행은 지우지 않습니다.
- 검수 전 행은 실행마다 새로 씁니다. **검수된 행이 하나라도 있는 블록**은 새 후보를 넣지 않습니다(재실행 중복 방지). 그 블록을 다시 보려면 검수된 행을 먼저 정리해야 합니다.
- `run_id`로 어느 실행에서 나온 행인지 확인합니다. 블록 오류는 `status: error`로 따로 남깁니다. 단, 검수된 행이 있는 블록의 오류는 다른 새 행과 함께 건너뜁니다.

In [6]:
import json, os, pathlib
OUT_STEPS = pathlib.Path("../../data/interim/prevention_legal/steps_raw_v2.jsonl")
OUT_STEPS.parent.mkdir(parents=True, exist_ok=True)

# 1) 사람이 검수한(approved/rejected) 행은 지우지 않는다. 검수 전(unreviewed) 행은 이번 실행 결과로 대체한다.
kept = []
if OUT_STEPS.exists():
    for line in OUT_STEPS.read_text(encoding="utf-8").splitlines():
        if line.strip():
            old = json.loads(line)
            if old.get("review_status") in ("approved", "rejected"):
                kept.append(old)

# 2) 검수된 행이 하나라도 있는 블록은 새 후보를 넣지 않는다 (재실행 때 같은 행동이 다른 문장으로 중복되는 것을 막는다)
reviewed_blocks = {r["chunk_id"] for r in kept}
new_rows = []
n_skip = 0
for r in rows:
    if r["chunk_id"] in reviewed_blocks:
        n_skip += 1
        continue
    new_rows.append(dict(r, model="gpt-6-luna", prompt_version="v2"))

# 임시 파일에 전부 쓴 뒤 한 번에 바꿔 끼운다 (쓰는 도중 실패해도 기존 파일이 깨지지 않도록)
TMP_STEPS = OUT_STEPS.with_suffix(".jsonl.tmp")
with TMP_STEPS.open("w", encoding="utf-8") as f:
    for r in kept + new_rows:
        f.write(json.dumps(r, ensure_ascii=False) + chr(10))
os.replace(TMP_STEPS, OUT_STEPS)

n_ok = sum(1 for r in new_rows if r["status"] == "ok")
n_err = sum(1 for r in new_rows if r["status"] == "error")
print(f"저장: {OUT_STEPS.name} | 보존(검수됨) {len(kept)}개 | 새 후보 {n_ok}개 | 검수된 블록이라 건너뜀 {n_skip}개 | 오류 {n_err}개")


저장: steps_raw_v2.jsonl | 보존(검수됨) 0개 | 새 후보 3개 | 검수된 블록이라 건너뜀 0개 | 오류 0개


## 9. 사람 검수 체크리스트
- [ ] 학교 행동이 맞는가? (의료기관·보건소 행동을 넣지 않았는가)
- [ ] 역할은 소제목에 근거하는가? (근거 없으면 null이어야 함)
- [ ] 강도(modality)가 '반드시' 같은 표현을 법적 의무로 과장하지 않았는가
- [ ] 근거 문장을 원문과 직접 대조했는가 (코드 확인과 별개)
- [ ] 93쪽 3번(유사증상자 2주 이내 2명 → 보건소 신고)이 학교 행동으로 잡혔는가

검수 후 `review_status`를 `approved` 또는 `rejected`로 바꿉니다.